# Reconstruction de synonymes à partir de WOLF

> **Origine** : ce notebook est adapté du TD de correction « Reconstruction de synonymes » de **Xavier Dupré** ([sdpython.github.io](https://sdpython.github.io/), ressources sur [xavierdupre.fr](http://www.xavierdupre.fr/)). Il n'a pas été écrit par l'équipe du projet ; il est diffusé sous licence MIT (voir `THIRD_PARTY_NOTICES.md`). Il explore une piste parallèle au graphe principal : retrouver automatiquement des synonymes dans un texte.

# Reconstruction de synonymes - correction

On s'intéresse la construction automatiquement de synonymes. En résumé :

* les synonymes via la base [WOLF](http://alpage.inria.fr/~sagot/wolf-en.html)
* on ne garde que les synonymes composé d'un seul mot
* on prend ensuite un texte quelconque qu'on découpe en phrase
* pour chaque phrase qu'on rencontre, on remplace des mots au hasard par leur synonymes
* chaque phrase sera présente une fois à l'identique et plusieurs fois avec des synonymes différents

L'objectif est alors de proposer une méthode pour reconstruire la base de synonymes.

## Génération des données

### Construction de la base de synonymes

La base [WOLF](http://alpage.inria.fr/~sagot/wolf-en.html) (WordNet libre du français) doit être téléchargée et placée à la racine du dépôt sous le nom `wolf-1.0b4.xml` (voir le `README`).

In [ ]:
import os
if not os.path.exists("../wolf-1.0b4.xml"):
    raise FileNotFoundError("../wolf-1.0b4.xml")
if os.stat("../wolf-1.0b4.xml").st_size < 3000000:
    raise FileNotFoundError("Size of 'wolf-1.0b4.xml' is very small: {0}".format(os.stat("../wolf-1.0b4.xml").st_size))

Fonction qui énumère, pour chaque synset de WOLF, la liste de ses synonymes français (lecture en flux : le fichier fait 40 Mo).

In [ ]:
import xml.etree.ElementTree as ET

def enumerate_wolf_synonyms(path):
    """Énumère les listes de synonymes (un synset = une liste de mots)."""
    for _, elem in ET.iterparse(path, events=("end",)):
        if elem.tag == "SYNSET":
            synonymes = [l.text for l in elem.iter("LITERAL") if l.text and l.text != "_EMPTY_"]
            if synonymes:
                yield synonymes
            elem.clear()

On passe en revue toute la basse (il y a environ 120.000 lignes) et on s'arrête après 10000 synonymes car sinon, cela prend un temps fou.

In [ ]:
allsyn = {}
for line, syn in enumerate(enumerate_wolf_synonyms("../wolf-1.0b4.xml")):
    if line % 10000 == 0: print("line", line, "allsyn", len(allsyn))
    clean = [_.lower() for _ in syn if " " not in _]
    if len(clean) > 1:
        for word in clean:
            if word not in allsyn:
                allsyn[word] = set(clean)
                continue
            else:
                for cl in clean:
                    allsyn[word].add(cl)
    if len(allsyn) > 10000:
        break
len(allsyn)

On affiche les premiers groupes :

In [ ]:
i = 0
for k, v in allsyn.items():
    print(k,v)
    i += 1
    if i > 10:
        break

### Génération d'une base de phrases modifiées

On utilise [Zadig](https://fr.wikipedia.org/wiki/Zadig).

In [ ]:
import os
import urllib.request

# Le texte est téléchargé une seule fois, puis conservé dans ../data/ (ignoré par Git)
zadig_path = "../data/zadig.txt"
if not os.path.exists(zadig_path):
    with urllib.request.urlopen("https://www.gutenberg.org/cache/epub/4647/pg4647.txt") as u:
        with open(zadig_path, "wb") as f:
            f.write(u.read())
with open(zadig_path, "rb") as f:
    content = f.read()

In [ ]:
char = content.decode(encoding="utf-8")

On découpe en mot.

In [ ]:
import re
reg = re.compile("([- a-zA-Zâàäéèëêîïôöùûü']+)")
phrases = [_.lower() for _ in reg.findall(char)]

In [ ]:
for i, phrase in enumerate(phrases):
    if i >= 990:
        print(phrase)
    if i >= 1000:
        break

On génère les phrases modifiées :

In [ ]:
import random
def modification(phrase, allsyn, nmax=10):
    mots = phrase.split()
    options = []
    nb = 1
    for mot in mots:
        if mot in allsyn:
            options.append(list(set([mot] + list(allsyn[mot]))))
        else:
            options.append([mot])
        nb *= len(options[-1])
    if nb == 1:
        return [phrase]
    else:
        res = []
        for i in range(0, min(nmax, nb//2+1, nb)):
            sol = []
            for mot in options:
                h = random.randint(0, len(mot)-1)
                sol.append(mot[h])
            res.append(sol)
        return res
            
modification("chatouiller le cérébral", allsyn)

On traite tous les mots :

In [ ]:
len(phrases)

In [ ]:
with open("../data/zadig_augmente.txt", "w", encoding="utf-8") as f:
    total = 0
    init = 0
    for i, phrase in enumerate(phrases):
        augm = modification(phrase, allsyn)
        init += 1
        for au in augm:
            f.write(" ".join(au) + "\n")
            total += 1
"total", total, "initial", init

## Exercice : retrouver une partie des synonymes à partir du fichier créé

Le fichier `../data/zadig_augmente.txt` est celui généré par le code précédent.

On lit le texte pour voir un peu de qu'il contient :

In [ ]:
with open("../data/zadig_augmente.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()
len(lines)

In [ ]:
lines[10000:10005]

On supprime les caractères indésirables avec la fonction [strip](https://docs.python.org/3.4/library/stdtypes.html#str.strip) :

In [ ]:
lines = [ line.strip("\n \t\r") for line in lines ]
lines[10000:10005]

Pour détecter les synonymes, on imagine le processus suivant, qu'on pourra améliorer au vu des premiers résultats :

1. on compare deux phrases consécutives du fichier ;
2. si elles ont beaucoup de mots en commun (au moins 50 %) et qu'**un seul mot** diffère de chaque côté, ces deux mots sont des candidats synonymes ;
3. on compte le nombre de fois où chaque paire apparaît : les paires les plus fréquentes sont les plus probables.

On supprime les phrases qui incluent des espaces entre chaque lettre. On a la choix entre les garder, les jeter ou les corriger. On choisit l'option la plus simple : les jeter quitte à y revenir plus tard. Si un espace existe en chaque lettre, cela veut que le nombre de lettres (sans espace) est égal au nombre de mot.

In [ ]:
garde = []
for line in lines:
    nbchar = len(line.replace(" ", ""))
    nbword = len(line.split())
    if nbchar > nbword:
        garde.append(line)
len(garde), len(lines)

In [ ]:
garde[:5]

On découpe en mot avec la fonction [split](https://docs.python.org/3.4/library/stdtypes.html#str.split) :

In [ ]:
mots = [line.split() for line in garde]
mots[:5]

On code les petits bouts :

In [ ]:
def pourcentage_mots_communs(p1, p2):
    n = 0
    for p in p1:
        if p in p2:
            n += 1
    return n * 1.0 / max(len(p1),len(p2))

p1, p2 = mots[0:2]
print(p1)
print(p2)
pourcentage_mots_communs(p1, p2)

Et l'autre :

In [ ]:
def mots_differents(p1, p2):
    diff1 = []
    for p in p1:
        if p not in p2:
            diff1.append(p)
    diff2 = []
    for p in p2:
        if p not in p1:
            diff2.append(p)
    return diff1, diff2

#p1, p2 = mots[0:2]
p1, p2 = ["a", "a", "b", "c"], ["a", "f", "g"]
print(p1)
print(p2)
mots_differents(p1, p2)

Pour la boucle final, on procède par étape, on affiche des résultats intermédiaires : 

In [ ]:
for i in range(0,len(mots)):
    p1 = mots[i]
    p2 = mots[i+1]
    print("----")
    print(p1)
    print(p2)
    print(pourcentage_mots_communs(p1,p2))
    print(mots_differents(p1,p2))
    if i > 3 :
        break

On complète cette première boucle en supposant qu'une bonne paire de synonymes doit provenir d'une paire de mots ayant en pourcentage de mots communs supérieur à 50% et une seule paire de synonymes :

In [ ]:
compteur = {}
for i in range(0,len(mots)-1):
    p1 = mots[i]
    p2 = mots[i+1]
    if pourcentage_mots_communs(p1, p2) >= 0.5:
        diff1, diff2 = mots_differents(p1, p2)
        if len(diff1) == len(diff2) == 1:
            m1 = diff1[0]
            m2 = diff2[0]
            l = [m1, m2]
            l.sort()  # on trie les deux mots pour aggréger les deux paires m1, m2 et m2, m1
            key = tuple(l)  # une liste ne peut être la clé d'un dictionnaire
            if key not in compteur:
                compteur[key] = 0
            compteur[key] += 1

On regarde le résultat en triant par ordre de fréquence décroissante :

In [ ]:
l = [ (compteur[k],k) for k in compteur]
l[:5]

In [ ]:
l.sort(reverse=True)
l[:20]

## Vérifications

Ces mots ne paraissent pas très pertinents au regard de la langue française. La méthode est néanmoins valable si elle permet de retrouver des synonymes qui ont permis de générer ce texte *zadig_augmente.txt*.

In [ ]:
allsyn.get("lombric", None)

In [ ]:
allsyn.get("flétan", None)

In [ ]:
allsyn.get("niet", None)

## Améliorations

### stop words

In [ ]:
stopwords = {"de", "ou", "a", "un", "du"}
compteur = {}
for i in range(0,len(mots)-1):
    p1 = mots[i]
    p2 = mots[i+1]
    if pourcentage_mots_communs(p1, p2) >= 0.5:
        diff1, diff2 = mots_differents(p1, p2)
        if len(diff1) == len(diff2) == 1:
            m1 = diff1[0]
            m2 = diff2[0]
            if m1 not in  stopwords and m2 not in stopwords:
                l = [m1, m2]
                l.sort()
                key = tuple(l)
                if key not in compteur:
                    compteur[key] = 0
                compteur[key] += 1

In [ ]:
l = [ (compteur[k],k) for k in compteur]
l.sort(reverse=True)
l[:20]

### prise en compte de plus ligne

Et pas seulement deux lignes consécutives

In [ ]:
stopwords = {"de", "ou", "a", "un", "du"}
compteur = {}
for i in range(0,len(mots)-1):
    p1 = mots[i]
    for k in range(0,10):
        if i+k >= len(mots):
            break
        p2 = mots[i+k]
        if pourcentage_mots_communs(p1, p2) >= 0.5:
            diff1, diff2 = mots_differents(p1, p2)
            if len(diff1) == len(diff2) == 1:
                m1 = diff1[0]
                m2 = diff2[0]
                if m1 not in  stopwords and m2 not in stopwords:
                    l = [m1, m2]
                    l.sort()
                    key = tuple(l)
                    if key not in compteur:
                        compteur[key] = 0
                    compteur[key] += 1

In [ ]:
l = [ (compteur[k],k) for k in compteur]
l.sort(reverse=True)
l[:20]

Ca ne change pas grand compte tenu de la façon dont la base a été construite.